# ADNI — folds 0/1/2 × seeds 0/1/4, no augmentation

Reuse every fully verified seed 0/1 result and resume interrupted seed 0/1 work first.
Run missing seed 0/1 combinations, then all missing seed 4 combinations. No seed 2 task is launched.
There are 504 results in the combined target; if seeds 0/1 are complete, only 168 new runs are needed.
Original seed 2 results and recovery remain untouched in the original folder.

**Before resuming:** the checked controller has already stopped. If you started another controller
since that check, stop it once and wait for verified recovery uploads. Do not run
this notebook simultaneously with the old notebook. It does not stop or delete the old runtime.
Upload only `adni_seeds014_resume_initfix_v2_code.zip` to the existing
`My Drive/hippo_adni_3fold_3seed/input/`. Keep the existing data ZIP, original code ZIP and results.
Mount the Google account containing those ADNI files. No dataset upload is needed.

Use the original validated A100-SXM4-80GB and software versions. All saved subject splits,
5% training data, initialization procedure, losses, calibration, BF16, batch size 1, optimizers,
learning-rate schedules, update counts and 31,976,450,667,090 optimization-FLOP cap remain unchanged.
nnUNet retains native Dice+CE/SGD and 60% targets; the other models retain Dice/AdamW and 10% targets.
**Absolutely no augmentation.** Seed 4 initialization has been exported on CPU and matched against
the completed cluster seed-4 controls. Those cluster results are provenance, not imported Colab runs.

This is a **post-selection exploratory comparison**: seed 2 was replaced after observing scores.
Keep the original seeds-0/1/2 report for transparent reporting. Do not describe this as an unbiased seed sample.


**Initialization repair v2:** replace only this notebook and upload the matching v2 code ZIP to
the existing input folder. Keep the original code ZIP, data ZIP, both results folders and recovery
files. Existing scientific source and manifests are byte-identical. The external worker uses a more
accurate CPU inverse-CDF calculation before the ORIGINAL strict check copies canonical Swin weights.
No tolerance is relaxed and no initialization/RNG/hash gate is bypassed.

The last checked snapshot had 329 completed target runs and four saved recoveries. These are not
hardcoded counts: every completed archive is checked and skipped; interrupted work is restored first.
Seven seeds-0/1 runs and 168 seed-4 runs remained at that snapshot. No new experiments are added.
After a clean stop or the confirmed drained failure, run this notebook from the top. On a fresh VM,
setup copies the existing data ZIP from Drive to local SSD; no dataset re-upload is needed.
Do not delete or reset any results, recovery, accounting, STOP or ownership files.


## 1. Mount Drive and stage the pinned code; preserve existing work


In [ ]:
from pathlib import Path
from contextlib import contextmanager, ExitStack
import fcntl, hashlib, json, os, shutil, signal, subprocess, sys, threading, time, zipfile
@contextmanager
def phase(label):
    print('PHASE:', label, flush=True)
    stop=threading.Event(); started=time.monotonic()
    def pulse():
        while not stop.wait(30):
            value=dict(phase=label, elapsed=time.monotonic()-started, status='still waiting; heartbeat is not progress')
            Path('/content/adni_seeds014_setup_status.json').write_text(json.dumps(value))
            print('HEARTBEAT:', value, flush=True)
    thread=threading.Thread(target=pulse,daemon=True);thread.start()
    try: yield
    finally: stop.set();thread.join()
from google.colab import drive
with phase('mount Drive'): drive.mount('/content/drive')
DRIVE_ROOT=Path('/content/drive/MyDrive/hippo_adni_3fold_3seed')
INPUT=DRIVE_ROOT/'input'
ORIGINAL_RESULTS=DRIVE_ROOT/'results_noaug_v1'
RESULTS=DRIVE_ROOT/'results_seeds014_noaug_v1'
ORIGINAL_BUNDLE=Path('/content/hippo_adni_bundle')
BUNDLE=Path('/content/hippo_adni_seed4_bundle')
PACKAGE=Path('/content/hippo_adni_seeds014_initfix_v2')
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024**2),b''):h.update(block)
    return h.hexdigest()
def stage_zip(name, expected, destination):
    local=Path('/content')/name
    with phase('copy '+name+' from Drive to local SSD'):
        shutil.copyfile(INPUT/name,local)
    assert sha256(local)==expected, 'ZIP hash mismatch: '+name
    with zipfile.ZipFile(local) as z:
        assert z.testzip() is None
        for i in z.infolist():
            p=Path(i.filename)
            assert not p.is_absolute() and '..' not in p.parts and ((i.external_attr>>16)&0o170000)!=0o120000
        z.extractall(destination)
def verify_files(root, manifest):
    for name, expected in json.loads(manifest.read_text()).items():
        assert sha256(root/name)==expected, 'Changed file: '+name
with phase('check original results and ownership'):
    assert (ORIGINAL_RESULTS/'environment_binding.json').is_file(), 'Mount the account containing your existing ADNI campaign.'
    for folder in (ORIGINAL_RESULTS,RESULTS):
        assert not (folder/'.adni_controller_lock').exists(), 'Existing controller ownership requires inspection; do not delete it.'
    assert not (RESULTS/'.adni_seeds014_controller_lock').exists(), 'Existing seeds014 ownership requires inspection.'
with ExitStack() as locks:
    for root in (ORIGINAL_BUNDLE,BUNDLE):
        (root/'work').mkdir(parents=True,exist_ok=True)
        lock=locks.enter_context((root/'work/sweep.lock').open('a'))
        fcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)
    for cmdline in Path('/proc').glob('[0-9]*/cmdline'):
        try: words=cmdline.read_bytes().decode(errors='replace').split('\0')
        except (FileNotFoundError,PermissionError,ProcessLookupError):continue
        for root in (ORIGINAL_BUNDLE,BUNDLE):
            assert not (str(root/'entry.py') in words or (str(root) in words and any(w.endswith(('/worker.py','/phase.py')) for w in words))), 'An ADNI worker is still active.'
    with phase('verify or stage original scientific bundle'):
        marker=ORIGINAL_BUNDLE/'bundle_manifest.json'
        if marker.exists():
            assert sha256(marker)=='e0b87c00231bb355aa66531f6684c361a632b3f8e393f6afbfb4939a37d0caeb'
            verify_files(ORIGINAL_BUNDLE,marker)
        else:
            assert not any((ORIGINAL_BUNDLE/'work').glob('*/budget_ledger.json')), 'Preserve and inspect existing work without manifest.'
            stage_zip('adni_replication_code.zip','1a1c1361f371e4b2f76f878f298aa4ca884627a4352c399e4b854daa79bfb436',ORIGINAL_BUNDLE)
            stage_zip('adni_data_verified.zip','e05bb92b0a6f5024dc9c823283a63020dba8f9cbef70141949a854f6bad6e709',ORIGINAL_BUNDLE)
            assert sha256(marker)=='e0b87c00231bb355aa66531f6684c361a632b3f8e393f6afbfb4939a37d0caeb'
            verify_files(ORIGINAL_BUNDLE,marker)
    stage_zip('adni_seeds014_resume_initfix_v2_code.zip','38598c0b56d617d31daa238fcd2d3677a324b8258d2d8bfee5a7bf2d4915e140',PACKAGE)
    verify_files(PACKAGE,PACKAGE/'package_manifest.json')
    with phase('stage separate seed4 source; reuse local dataset'):
        marker=BUNDLE/'bundle_manifest.json'
        if marker.exists():
            assert sha256(marker)=='88349a82b03b4f93884def610f5ef12077968ef3b1b75192a3472f395266a148'
            verify_files(BUNDLE,marker)
        else:
            assert not any((BUNDLE/'work').glob('*/budget_ledger.json')), 'Preserve seed4 work without manifest.'
            shutil.copytree(PACKAGE/'seed4_bundle',BUNDLE,dirs_exist_ok=True)
            (BUNDLE/'data').mkdir(exist_ok=True)
            data=BUNDLE/'data/adni_hippocampus_full.pkl'
            if not data.exists():data.symlink_to(ORIGINAL_BUNDLE/'data/adni_hippocampus_full.pkl')
            verify_files(BUNDLE,marker)
RESULTS.mkdir(parents=True,exist_ok=True)
print('Seeds 0/1 results stay in:',ORIGINAL_RESULTS)
print('Seed4 and combined comparison:',RESULTS)


## 2. Authenticate quota checks and prepare the same training environment


In [ ]:
from google.colab import auth
with phase('authenticate Drive quota API'):auth.authenticate_user()
VENV=Path('/content/hippo_adni_venv');PYTHON=VENV/'bin/python'
def run_logged(args, name):
    path=Path('/content')/('adni_seeds014_'+name+'.log')
    with path.open('a') as log:
        p=subprocess.Popen(list(map(str,args)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
        def drain():
            for line in p.stdout:print(line,end='',flush=True);log.write(line);log.flush()
        interrupted=False
        try:drain();code=p.wait()
        except KeyboardInterrupt:
            interrupted=True
            print('Requesting clean stop; wait for all verified recovery uploads.',flush=True)
            if name in ('run','report'):p.send_signal(signal.SIGINT)
            else:os.killpg(p.pid,signal.SIGINT)
            drain();code=p.wait()
        finally:p.stdout.close()
        if interrupted:raise KeyboardInterrupt('Stopped cleanly; rerun later to resume.')
        if code:raise RuntimeError(f'Exit {code}; inspect {path}. Recovery preserved.')
with phase('prepare training environment'):
    if not PYTHON.exists():run_logged([sys.executable,'-m','venv','--without-pip','--system-site-packages',VENV],'setup')
    if subprocess.run([str(PYTHON),'-m','pip','--version'],capture_output=True).returncode:
        import urllib.request
        bootstrap=Path('/content/hippo_adni_get_pip.py')
        urllib.request.urlretrieve('https://bootstrap.pypa.io/get-pip.py',bootstrap)
        run_logged([PYTHON,bootstrap],'setup')
    run_logged([PYTHON,'-m','pip','install','--disable-pip-version-check','-r',ORIGINAL_BUNDLE/'requirements-colab.txt'],'setup')
COMMON=['--parent-bundle',ORIGINAL_BUNDLE,'--parent-results',ORIGINAL_RESULTS,'--bundle',BUNDLE,'--durable',RESULTS]
def campaign(command,*args):
    run_logged([PYTHON,'-u',PACKAGE/'campaign.py',command,*COMMON,*args],command)
subprocess.run(['nvidia-smi'],check=True)


## 3. Required compatibility preflight (CPU only; no extra experiments)

Verify the unchanged source, saved GPU/software bindings, original passed GPU preflights and
A100 acceleration evidence. Then check Swin initializations for seeds 0/1/4 on CPU: the original
strict tolerance, all RNG checks and exact complete model hashes must pass. The same initialization
hashes are required in all three folds. This does not run forward/backward passes, optimizer updates
or benchmarks, and does not replace the original preflight receipts. A failed check blocks training.
Use the saved A100-SXM4-80GB environment; a different GPU/software environment is not silently accepted.


In [ ]:
with phase('verify saved preflights and CPU Swin initialization; no training trials'):
    for root, durable, control in (
        (ORIGINAL_BUNDLE, ORIGINAL_RESULTS, PACKAGE/'control01'),
        (BUNDLE, RESULTS, PACKAGE/'control04'),
    ):
        run_logged([PYTHON, '-u', PACKAGE/'preflight_initfix.py',
                    '--bundle', root, '--durable', durable, '--control', control], 'preflight')


## 4. Resume missing seeds0/1, then run missing seed4

Full archive verification runs before any launches. Checked/verified counts and live phases
appear every30seconds, with local status independent of Drive. The scheduler fills the CPU-supported
worker target subject to memory/storage/baseline dependencies; no profiling trials are added.
Both phases share one session deadline and launch limit. Every run keeps the original charge-before-update
ledger, recovery uploads, FLOP cap and archive-before-cleanup checks. Seed2 is never launched or removed.
A phase failure prevents the next phase. Original and new STOP files are respected, never cleared.

Interrupt once to stop cleanly and wait for all uploads. Use the same notebook and folders to resume.
Set SESSION_HOURS below the time remaining in your current Colab runtime;22hours is only a conservative
target for a newly allocated runtime, not a guarantee. The default8hours allows a handover.


In [ ]:
MAX_NEW_RUNS=504  # Maximum missing tasks, never repeats verified results.
MAX_WORKERS=0    # CPU-supported target; actual occupancy depends on resource checks.
SESSION_HOURS=8.0
campaign('run','--max-runs',MAX_NEW_RUNS,'--workers',MAX_WORKERS,'--hours',SESSION_HOURS)


## 5. Combined comparison for seeds0/1/4

This cell re-verifies archives but starts no training. All nine model/condition cells are required
for a final mean: average seeds within each fold, then average the three folds. Gains use the exact
matched model/fold/seed baseline. The original comparison.json is preserved; combined results live
in results_seeds014_noaug_v1/comparison.json. Seed2 is excluded from this view but remains archived.
Architecture/loss/optimizer confounding and the post-selection seed change must remain explicit.


In [ ]:
campaign('report','--hours',2)
import pandas as pd
result=json.loads((RESULTS/'comparison.json').read_text())
print(f"Verified requested results: {result['completed']}/{result['total']}")
display(pd.DataFrame(result['aggregate']))
display(pd.DataFrame(result['runs']))


Local live status: /content/hippo_adni_bundle/parallel_status and
/content/hippo_adni_seed4_bundle/parallel_status. Notebook logs are /content/adni_seeds014_*.log.
Original results/recovery remain in results_noaug_v1; new seed4 results/recovery and combined tables
are in results_seeds014_noaug_v1. Existing completed runs are never migrated or rewritten.
No further dataset upload is needed after a VM reset: rerun setup to stage the existing Drive ZIPs.
Do not delete ownership locks after an unclean termination; inspect recovery/accounting first.
